# Volume Prediction with DeepLOB Encoder

This notebook predicts per-second **OHLCV volume** in the last 60 seconds using a DeepLOB-based encoder.

**Goal:** Empirically validate the oracle analysis in `volume_prediction_analysis.md` — that volume prediction
at per-second granularity is difficult and provides limited value for execution optimization.

**4 improvements over naive approach (which scored R² = -21.67):**
1. **Log-transform volume** — `log1p(volume)` before normalization to compress heavy tails
2. **Non-autoregressive head** — Linear projection from encoder instead of Seq2Seq decoder (volume is not autocorrelated like price)
3. **Huber loss** — Robust to volume spikes, unlike MSE which is dominated by outliers
4. **No midprice seed** — Subsumed by fix 2; the non-autoregressive head eliminates the decoder seed entirely

In [1]:
import os, sys, subprocess

# Get project files onto remote runtime (Colab extension doesn't sync files)
if not os.path.isdir("/content/Ultramarin/utils"):
    subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                    "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                   cwd="/content")
os.chdir("/content/Ultramarin")
sys.path.insert(0, os.getcwd())

import copy
import numpy as np
from pathlib import Path
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import random
from utils.utils import compute_ofi
from data.simulate_walk_the_book import simulate_walk_the_book
from utils.datastuff import TrainCfg, LOBProcessor, TensorTimeDataset
from utils.train import chrono_split
from models.DeepLOB import DeepLOBEncoder
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device: cuda


In [2]:
# Copy data from Google Drive (data/ is gitignored, parquets don't sync)
import os, subprocess

dst = "/content/Ultramarin/data"

if not os.path.isfile(os.path.join(dst, "BTCUSDT/X_train.parquet")):
    print("Data not found. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

    src = "/content/drive/MyDrive/data"
    os.makedirs(dst, exist_ok=True)
    print(f"Copying from {src} to {dst} ...")

    result = subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR copying data: {result.stderr}")
    else:
        print("Data copied successfully.")

    print(f"data/ contents: {os.listdir(dst)}")
    btc_dir = os.path.join(dst, "BTCUSDT")
    if os.path.isdir(btc_dir):
        print(f"data/BTCUSDT/ contents: {os.listdir(btc_dir)}")
else:
    print("Data already present.")

Data already present.


In [3]:
# Paths and volume_to_fill
from pathlib import Path

root = Path("data") if Path("data").exists() else Path.cwd()

data_asset = "BTCUSDT"
symbol_dir = root / data_asset
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
X_test_path = symbol_dir / "X_test.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

# Optimal K per currency (from all-currency K-sweep on full training data)
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}
K_SECONDS = OPTIMAL_K.get(data_asset, 14)

# Known volumes per currency (fallback if vol_to_fill.txt is missing)
KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))

if volume_to_fill is None:
    volume_to_fill = KNOWN_VOLUMES.get(data_asset)
    print(f"vol_to_fill.txt not found, using known volume: {volume_to_fill}")
else:
    print(f"volume_to_fill: {volume_to_fill}")

print(f"K_SECONDS: {K_SECONDS} (optimal for {data_asset})")

volume_to_fill: 4.0
K_SECONDS: 14 (optimal for BTCUSDT)


In [4]:
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

OFI_COLS = ['ofi_1', 'ofi_2', 'ofi_3', 'ofi_4', 'ofi_5', 'ofi_agg']

FEATURE_COLS = LOB_COLS + ["mid_price"] + OFI_COLS + ["volume"]

print(f"LOB Features: {len(LOB_COLS)}")
print(f"Extra Features: mid_price + {len(OFI_COLS)} OFI + volume = {1 + len(OFI_COLS) + 1}")
print(f"Total Features: {len(FEATURE_COLS)} (20 LOB + 1 mid_price + 6 OFI + 1 volume)")

LOB Features: 20
Extra Features: mid_price + 6 OFI + volume = 8
Total Features: 28 (20 LOB + 1 mid_price + 6 OFI + 1 volume)


In [5]:
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# Model Definition

**VolumeModel** uses the same DeepLOB encoder + biLSTM as `SuperModel`, but with a key change:

**Non-autoregressive head** — replaces the Seq2Seq decoder with a linear projection.
Volume is not autocorrelated at per-second granularity, so step-by-step generation compounds errors.
The encoder output is mean-pooled over time and projected to 60 per-second volume predictions.

In [6]:
class VolumeModel(nn.Module):
    """DeepLOB encoder + biLSTM + non-autoregressive linear head for volume prediction."""

    def __init__(self, hidden=128, horizon=60, dropout=0.1, num_extra_features=0):
        super().__init__()
        self.spatial = DeepLOBEncoder(in_ch=1, dropout=dropout)
        encoder_input_size = 192 + num_extra_features
        self.encoder = nn.LSTM(
            input_size=encoder_input_size,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.enc_dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, horizon),  # Per-second volume prediction
        )
        self.horizon = horizon
        self.num_extra_features = num_extra_features

    def forward(self, x, y_teacher=None, tf_ratio=0.0):
        # x: [B, T, F]
        lob_features = x[:, :, :20]
        h_spatial = self.spatial(lob_features)  # [B, T, 192]

        if self.num_extra_features > 0:
            extra_features = x[:, :, 20:]  # [B, T, N_extra]
            h_spatial = torch.cat([h_spatial, extra_features], dim=-1)

        enc_out, _ = self.encoder(h_spatial)  # [B, T, 2H]
        enc_out = self.enc_dropout(enc_out)

        # Mean-pool over encoder timesteps
        enc_summary = enc_out.mean(dim=1)  # [B, 2H]

        return self.head(enc_summary)  # [B, horizon]

print("VolumeModel defined.")

VolumeModel defined.


# Model Training

**4 improvements applied:**
1. `log1p(volume)` before normalization — compresses heavy-tailed distribution
2. `VolumeModel` with non-autoregressive head — no error compounding
3. Huber loss (delta=1.0) — robust to volume spikes
4. No decoder seed mismatch — the linear head has no autoregressive seed

In [7]:
config = TrainCfg(

    # Hyperparameters
    epochs = 100,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.0,
    direction_lambda = 0.0,
    dropout = 0.1,

    # Scheduled teacher forcing (not used by VolumeModel, kept for config compat)
    tf_floor = 0.1,

    # Early stopping
    patience = 10,
    min_lr = 1e-6,

    # Windows
    input_window = 600,
    target_window = 60,
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),

    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    x_test_path = X_test_path,
    feature_cols = FEATURE_COLS,
)

# --- 1. Load Data ---
X_raw = pd.read_parquet(config.x_path).sort_values(["anonymized_id", "time_in_hour"])
Y_raw = pd.read_parquet(config.y_path).sort_values(["anonymized_id", "time_in_hour"])

# --- 2. Chrono Split ---
x_train_df, x_val_df, y_train_df, y_val_df = chrono_split(X_raw, Y_raw, val_ratio=config.val_ratio)

# --- 3. Add mid_price ---
x_train_df = x_train_df.assign(mid_price=(x_train_df["ask_price_1"] + x_train_df["bid_price_1"]) / 2.0)
x_val_df   = x_val_df.assign(mid_price=(x_val_df["ask_price_1"]     + x_val_df["bid_price_1"])   / 2.0)
y_train_df = y_train_df.assign(mid_price=(y_train_df["ask_price_1"] + y_train_df["bid_price_1"]) / 2.0)
y_val_df   = y_val_df.assign(mid_price=(y_val_df["ask_price_1"]     + y_val_df["bid_price_1"])   / 2.0)

# --- FIX 1: Log-transform volume before normalization ---
# Compresses heavy tail: raw volume can spike 100x, log1p makes it ~5x
x_train_df = x_train_df.assign(volume=np.log1p(x_train_df["volume"].clip(lower=0)))
x_val_df   = x_val_df.assign(volume=np.log1p(x_val_df["volume"].clip(lower=0)))
y_train_df = y_train_df.assign(volume=np.log1p(y_train_df["volume"].clip(lower=0)))
y_val_df   = y_val_df.assign(volume=np.log1p(y_val_df["volume"].clip(lower=0)))
print("Applied log1p transform to volume column.")

# --- 4. Compute OFI features ---
x_train_df = compute_ofi(x_train_df)
x_val_df   = compute_ofi(x_val_df)
y_train_df = compute_ofi(y_train_df)
y_val_df   = compute_ofi(y_val_df)

# --- 5. Process with LOBProcessor ---
processor = LOBProcessor(config, device=config.device)

train_out = processor.process(pl.from_pandas(x_train_df), pl.from_pandas(y_train_df))
X_train_tensor, Y_train_tensor = train_out["X"], train_out["y"]

val_out = processor.process(pl.from_pandas(x_val_df), pl.from_pandas(y_val_df))
X_val_tensor, Y_val_tensor, val_id_map = val_out["X"], val_out["y"], val_out["y_id_map"]

# --- 6. Enforce FEATURE_COLS order and extract scalers ---
feature_indices = [processor.feature_map[col] for col in config.feature_cols]

X_train_tensor = X_train_tensor[:, :, feature_indices]
X_val_tensor   = X_val_tensor[:, :, feature_indices]
train_means    = train_out["means"][:, :, feature_indices]
train_stds     = train_out["stds"][:, :, feature_indices]

feature_index_map = {col: i for i, col in enumerate(config.feature_cols)}

# --- 7. Extract VOLUME as target ---
volume_idx = feature_index_map["volume"]
Y_train_tensor = Y_train_tensor[:, :, feature_indices]
Y_val_tensor   = Y_val_tensor[:, :, feature_indices]

vol_train = Y_train_tensor[:, :, volume_idx]  # [60, num_train_ids]
vol_val   = Y_val_tensor[:, :, volume_idx]    # [60, num_val_ids]

print(f"Volume target shape (train): {vol_train.shape}")
print(f"Volume target shape (val):   {vol_val.shape}")

# --- 8. Create datasets ---
train_dataset = TensorTimeDataset(X_train_tensor, Y_train_tensor, vol_train.T, input_window=config.input_window)
val_dataset   = TensorTimeDataset(X_val_tensor, Y_val_tensor, vol_val.T, input_window=config.input_window)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

# --- FIX 2: Non-autoregressive VolumeModel ---
num_extra_features = len(config.feature_cols) - 20
model = VolumeModel(
    horizon=config.target_window,
    dropout=config.dropout,
    num_extra_features=num_extra_features,
).to(config.device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# --- FIX 3: Huber loss ---
def volume_loss(pred, target):
    return nn.functional.huber_loss(pred, target, delta=1.0)

# --- Custom train/eval using Huber (can't reuse train_epoch which calls forecast_loss) ---
def train_volume_epoch(model, loader, optimizer, cfg):
    model.train()
    total_loss, num_samples = 0.0, 0
    for x_batch, y_batch, target in loader:
        x_batch, target = x_batch.to(cfg.device), target.to(cfg.device)
        optimizer.zero_grad()
        pred = model(x_batch)
        loss = volume_loss(pred, target)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * x_batch.size(0)
        num_samples += x_batch.size(0)
    return total_loss / max(num_samples, 1)

def eval_volume_epoch(model, loader, cfg):
    model.eval()
    total_loss, num_samples = 0.0, 0
    with torch.no_grad():
        for x_batch, _, target in loader:
            x_batch, target = x_batch.to(cfg.device), target.to(cfg.device)
            pred = model(x_batch)
            loss = volume_loss(pred, target)
            total_loss += float(loss.item()) * x_batch.size(0)
            num_samples += x_batch.size(0)
    return total_loss / max(num_samples, 1)

# --- 9. Training loop ---
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=config.min_lr)

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

for epoch in range(config.epochs):
    train_loss = train_volume_epoch(model, train_loader, optimizer, config)
    val_loss   = eval_volume_epoch(model, val_loader, config)
    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch+1:02d} | Train: {train_loss:.6f} | Val: {val_loss:.6f} | LR: {current_lr:.2e}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= config.patience:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {config.patience} epochs)")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"Restored best model (val loss: {best_val_loss:.6f})")

# --- 10. Save scalers for denormalization ---
val_means = val_out["means"][:, :, feature_indices]
val_stds  = val_out["stds"][:, :, feature_indices]

train_scalers = {
    "feat_means": train_means,
    "feat_stds": train_stds,
    "val_feat_means": val_means,
    "val_feat_stds": val_stds,
}

Applied log1p transform to volume column.
Volume target shape (train): torch.Size([60, 564])
Volume target shape (val):   torch.Size([60, 141])
Model parameters: 451,100
Epoch 01 | Train: 0.210995 | Val: 0.226489 | LR: 1.00e-03
Epoch 02 | Train: 0.205816 | Val: 0.226325 | LR: 9.99e-04
Epoch 03 | Train: 0.205354 | Val: 0.227036 | LR: 9.98e-04
Epoch 04 | Train: 0.204771 | Val: 0.226773 | LR: 9.96e-04
Epoch 05 | Train: 0.204637 | Val: 0.226155 | LR: 9.94e-04
Epoch 06 | Train: 0.204247 | Val: 0.225139 | LR: 9.91e-04
Epoch 07 | Train: 0.204215 | Val: 0.225180 | LR: 9.88e-04
Epoch 08 | Train: 0.203835 | Val: 0.224920 | LR: 9.84e-04
Epoch 09 | Train: 0.203695 | Val: 0.225045 | LR: 9.80e-04
Epoch 10 | Train: 0.203371 | Val: 0.225390 | LR: 9.76e-04
Epoch 11 | Train: 0.203205 | Val: 0.224888 | LR: 9.70e-04
Epoch 12 | Train: 0.203451 | Val: 0.225314 | LR: 9.65e-04
Epoch 13 | Train: 0.203365 | Val: 0.225020 | LR: 9.59e-04
Epoch 14 | Train: 0.202633 | Val: 0.224863 | LR: 9.52e-04
Epoch 15 | Train: 

# Volume Prediction Quality

Evaluate R² and MSE. Denormalization path: z-score undo -> expm1 (undo log1p) -> raw volume.

For context, mid-price R² is typically 0.90-0.95. Literature reports volume R² of 0.2-0.6 at **15-minute** intervals.

In [8]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch)
        all_preds.append(pred.cpu())
        all_targets.append(target.cpu())

all_preds = torch.cat(all_preds)      # [num_val_ids, 60]
all_targets = torch.cat(all_targets)  # [num_val_ids, 60]

# Denormalize: undo z-score -> undo log1p
vol_mean = train_scalers["val_feat_means"][:, :, volume_idx].squeeze(0).unsqueeze(1).cpu()
vol_std  = train_scalers["val_feat_stds"][:, :, volume_idx].squeeze(0).unsqueeze(1).cpu()

preds_log   = all_preds * vol_std + vol_mean       # log-space volume
targets_log = all_targets * vol_std + vol_mean      # log-space volume
preds_denorm   = torch.expm1(preds_log)             # raw volume
targets_denorm = torch.expm1(targets_log)           # raw volume

# Per-instrument R² (in raw volume space)
y_mean = targets_denorm.mean(dim=1, keepdim=True)
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)
r2_per_inst = 1 - ss_res / ss_tot

valid_r2 = r2_per_inst[ss_tot > 1e-6]

final_r2 = valid_r2.mean().item()
final_mse = ((preds_denorm - targets_denorm) ** 2).mean().item()

# Also compute R² in normalized (z-score) space for honest comparison
norm_y_mean = all_targets.mean(dim=1, keepdim=True)
norm_ss_res = ((all_targets - all_preds) ** 2).sum(dim=1)
norm_ss_tot = ((all_targets - norm_y_mean) ** 2).sum(dim=1)
norm_r2_per_inst = 1 - norm_ss_res / norm_ss_tot
norm_valid_r2 = norm_r2_per_inst[norm_ss_tot > 1e-6]
norm_r2 = norm_valid_r2.mean().item()

val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
val_preds = all_preds.numpy()

print(f"{'='*55}")
print(f"VOLUME PREDICTION QUALITY ({data_asset})")
print(f"{'='*55}")
print(f"Validation instruments: {len(val_ids)}")
print(f"Instruments with non-constant volume: {len(valid_r2)}")
print(f"")
print(f"--- Raw volume space ---")
print(f"Denormalized MSE:    {final_mse:.6f}")
print(f"Denormalized R²:     {final_r2:.6f}")
print(f"Median R² per inst:  {valid_r2.median().item():.6f}")
print(f"")
print(f"--- Normalized (z-score) space ---")
print(f"Normalized R²:       {norm_r2:.6f}")
print(f"Median norm R²:      {norm_valid_r2.median().item():.6f}")
print(f"")
print(f"R² > 0 (any signal): {(valid_r2 > 0).float().mean().item()*100:.1f}%")
print(f"R² > 0.3 (decent):   {(valid_r2 > 0.3).float().mean().item()*100:.1f}%")
print(f"R² > 0.5 (good):     {(valid_r2 > 0.5).float().mean().item()*100:.1f}%")
print(f"{'='*55}")
print(f"")
print(f"For comparison, mid-price R² is typically 0.90-0.95.")
print(f"Literature reports volume R² of 0.2-0.6 at 15-MINUTE intervals.")

VOLUME PREDICTION QUALITY (BTCUSDT)
Validation instruments: 141
Instruments with non-constant volume: 141

--- Raw volume space ---
Denormalized MSE:    0.216747
Denormalized R²:     -2.260182
Median R² per inst:  -0.036392

--- Normalized (z-score) space ---
Normalized R²:       -2.233174
Median norm R²:      -0.036385

R² > 0 (any signal): 9.2%
R² > 0.3 (decent):   0.0%
R² > 0.5 (good):     0.0%

For comparison, mid-price R² is typically 0.90-0.95.
Literature reports volume R² of 0.2-0.6 at 15-MINUTE intervals.


# Can Volume Predictions Improve Execution?

**VWAP-style approach:** Allocate more volume to seconds where predicted trading volume is high
(assumption: high volume = more liquidity = lower market impact).

In [9]:
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

vwap_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")

    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    # Denormalize: undo z-score -> undo log1p -> raw volume
    vol_pred_raw = np.expm1(val_preds[i] * vol_std[i].item() + vol_mean[i].item())

    # VWAP-style: trade proportionally to predicted volume in last K seconds
    vol_window = vol_pred_raw[-K_SECONDS:]
    weights = np.maximum(vol_window, 0)

    if weights.sum() > 1e-8:
        weights = weights / weights.sum()
    else:
        weights = np.ones(K_SECONDS) / K_SECONDS

    positions = np.zeros(60)
    positions[-K_SECONDS:] = weights * volume_to_fill

    vol, avg_price = simulate_walk_the_book(
        positions, ask_prices, ask_vols, bid_prices, bid_vols
    )

    if vol > 0 and not np.isnan(avg_price):
        impl_error = np.abs(avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / vol)
        vwap_bps.append(impl_error * vol_penalty)

vwap_bps = np.array(vwap_bps)

print(f"\n{'='*55}")
print(f"VOLUME-WEIGHTED (VWAP-STYLE) EXECUTION")
print(f"{'='*55}")
print(f"Window: last {K_SECONDS}s, weights proportional to predicted volume")
print(f"Instruments evaluated: {len(vwap_bps)}")
print(f"Mean:   {vwap_bps.mean():.4f} bps")
print(f"Median: {np.median(vwap_bps):.4f} bps")
print(f"Std:    {vwap_bps.std():.4f} bps")
print(f"{'='*55}")


VOLUME-WEIGHTED (VWAP-STYLE) EXECUTION
Window: last 14s, weights proportional to predicted volume
Instruments evaluated: 141
Mean:   1.3574 bps
Median: 1.1772 bps
Std:    1.1058 bps


In [10]:
# --- Blended Volume-Weighted + TWAP (10% model, 90% TWAP) ---

blended_bps = []
ALPHA = 0.1

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")

    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    vol_pred_raw = np.expm1(val_preds[i] * vol_std[i].item() + vol_mean[i].item())

    vol_window = vol_pred_raw[-K_SECONDS:]
    weights = np.maximum(vol_window, 0)
    if weights.sum() > 1e-8:
        weights = weights / weights.sum()
    else:
        weights = np.ones(K_SECONDS) / K_SECONDS

    model_positions = np.zeros(60)
    model_positions[-K_SECONDS:] = weights * volume_to_fill

    twap_positions = np.zeros(60)
    twap_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS

    blended_positions = ALPHA * model_positions + (1 - ALPHA) * twap_positions
    blended_positions = blended_positions * (volume_to_fill / blended_positions.sum())

    vol, avg_price = simulate_walk_the_book(
        blended_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )

    if vol > 0 and not np.isnan(avg_price):
        impl_error = np.abs(avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / vol)
        blended_bps.append(impl_error * vol_penalty)

blended_bps = np.array(blended_bps)

print(f"\n{'='*55}")
print(f"BLENDED VOLUME-WEIGHTED ({int(ALPHA*100)}% model / {int((1-ALPHA)*100)}% TWAP)")
print(f"{'='*55}")
print(f"Instruments evaluated: {len(blended_bps)}")
print(f"Mean:   {blended_bps.mean():.4f} bps")
print(f"Median: {np.median(blended_bps):.4f} bps")
print(f"{'='*55}")


BLENDED VOLUME-WEIGHTED (10% model / 90% TWAP)
Instruments evaluated: 141
Mean:   1.3259 bps
Median: 1.1781 bps


# TWAP Baseline

In [11]:
baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")

    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS

    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )

    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print(f"\n{'='*55}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*55}")
print(f"Instruments evaluated: {len(baseline_bps)}")
print(f"Mean:   {baseline_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_bps):.4f} bps")
print(f"Std:    {baseline_bps.std():.4f} bps")
print(f"{'='*55}")


BASELINE (TWAP-14s) IMPLEMENTATION ERROR
Instruments evaluated: 141
Mean:   1.3232 bps
Median: 1.1836 bps
Std:    1.0940 bps


# Summary Comparison

In [12]:
print(f"{'='*65}")
print(f"VOLUME PREDICTION EXPERIMENT SUMMARY ({data_asset})")
print(f"{'='*65}")
print(f"")
print(f"--- Improvements Applied ---")
print(f"1. Log-transform volume (log1p before normalization)")
print(f"2. Non-autoregressive head (VolumeModel replaces Seq2Seq decoder)")
print(f"3. Huber loss (robust to volume spikes)")
print(f"4. No decoder seed mismatch (subsumed by fix 2)")
print(f"")
print(f"--- Prediction Quality ---")
print(f"Normalized R²:     {norm_r2:.4f}  (was -21.67 before improvements)")
print(f"Denormalized R²:   {final_r2:.4f}")
print(f"Denormalized MSE:  {final_mse:.4f}")
print(f"(Mid-price R² is typically 0.90-0.95 with the same encoder)")
print(f"")
print(f"--- Execution Quality (BPS, lower is better) ---")
print(f"{'Strategy':<40} {'Mean BPS':>10} {'Median BPS':>12}")
print(f"{'-'*62}")
print(f"{'TWAP-' + str(K_SECONDS) + 's (baseline)':<40} {baseline_bps.mean():>10.4f} {np.median(baseline_bps):>12.4f}")
print(f"{'Volume-weighted (VWAP-style)':<40} {vwap_bps.mean():>10.4f} {np.median(vwap_bps):>12.4f}")
print(f"{'Blended (' + str(int(ALPHA*100)) + '% vol / ' + str(int((1-ALPHA)*100)) + '% TWAP)':<40} {blended_bps.mean():>10.4f} {np.median(blended_bps):>12.4f}")
print(f"")
print(f"--- Delta vs TWAP ---")
print(f"Volume-weighted vs TWAP: {vwap_bps.mean() - baseline_bps.mean():+.4f} bps")
print(f"Blended vs TWAP:        {blended_bps.mean() - baseline_bps.mean():+.4f} bps")
print(f"{'='*65}")

VOLUME PREDICTION EXPERIMENT SUMMARY (BTCUSDT)

--- Improvements Applied ---
1. Log-transform volume (log1p before normalization)
2. Non-autoregressive head (VolumeModel replaces Seq2Seq decoder)
3. Huber loss (robust to volume spikes)
4. No decoder seed mismatch (subsumed by fix 2)

--- Prediction Quality ---
Normalized R²:     -2.2332  (was -21.67 before improvements)
Denormalized R²:   -2.2602
Denormalized MSE:  0.2167
(Mid-price R² is typically 0.90-0.95 with the same encoder)

--- Execution Quality (BPS, lower is better) ---
Strategy                                   Mean BPS   Median BPS
--------------------------------------------------------------
TWAP-14s (baseline)                          1.3232       1.1836
Volume-weighted (VWAP-style)                 1.3574       1.1772
Blended (10% vol / 90% TWAP)                 1.3259       1.1781

--- Delta vs TWAP ---
Volume-weighted vs TWAP: +0.0342 bps
Blended vs TWAP:        +0.0026 bps


# Conclusions

This experiment validates the oracle analysis from `volume_prediction_analysis.md`:

1. **Volume R² is much lower than mid-price R², even with 4 targeted improvements.**
   Log-transform, non-autoregressive head, and Huber loss all helped vs the naive Seq2Seq
   approach (R² improved from -21.67), but volume at per-second granularity remains
   fundamentally harder than mid-price.

2. **Volume predictions do not improve execution.** Volume-weighted (VWAP-style) scheduling
   does not beat the TWAP baseline. This confirms Joseph's oracle finding: even with *perfect*
   volume knowledge, the improvement is only ~0.85 bps — with realistic prediction quality,
   it rounds to near zero.

3. **The ceiling is the bottleneck, not the model.** Even with architectural improvements
   tailored for volume, the fundamental issue remains: predicted volume has a weak relationship
   with execution cost.

**Decision:** Focus on mid-price prediction (where DeepLOB excels) and spread prediction
(where the oracle ceiling is 2x higher than volume).